In [ ]:
!pip install torch numpy pandas matplotlib tqdm scikit-learn


In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-04-10 09:56:16--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-04-10 09:56:16 (187 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open("input.txt", "r") as f:
    text = f.read()

print(len(text))

1115394


In [ ]:
from collections import Counter
import re

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\w+|[^\w\s]", text)  # keep punctuation
    return tokens

tokens = tokenize(text)

print(tokens[:30])  # check sample

['first', 'citizen', ':', 'before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'all', ':', 'speak', ',', 'speak', '.', 'first', 'citizen', ':', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than']


In [ ]:
# ✅ NEW (CORRECT — fixes CUDA error)

from collections import Counter

MAX_VOCAB = 5000

vocab = Counter(tokens)
vocab = vocab.most_common(MAX_VOCAB)

# indexing starts from 0
word2idx = {word: i for i, (word, _) in enumerate(vocab)}

# add pad token at end
word2idx["<pad>"] = len(word2idx)

idx2word = {i: word for word, i in word2idx.items()}

vocab_size = len(word2idx)

print("Vocab size:", vocab_size)

Vocab size: 5001


In [ ]:
encoded = [word2idx[word] for word in tokens if word in word2idx]

print(encoded[:20])

[100, 283, 1, 152, 40, 985, 158, 678, 0, 137, 24, 114, 2, 43, 1, 114, 0, 114, 2, 100]


In [ ]:
import torch

SEQ_LEN = 32

def create_sequences(data, seq_len):
    inputs = []
    targets = []

    for i in range(len(data) - seq_len):
        inputs.append(data[i:i+seq_len])
        targets.append(data[i+1:i+seq_len+1])

    return torch.tensor(inputs), torch.tensor(targets)

inputs, targets = create_sequences(encoded, SEQ_LEN)

print(inputs.shape, targets.shape)

torch.Size([254882, 32]) torch.Size([254882, 32])


In [ ]:
total = inputs.shape[0]

train_end = int(0.7 * total)
val_end = int(0.85 * total)

train_x, train_y = inputs[:train_end], targets[:train_end]
val_x, val_y = inputs[train_end:val_end], targets[train_end:val_end]
test_x, test_y = inputs[val_end:], targets[val_end:]

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 32

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(
    TensorDataset(val_x, val_y),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
batch_input, batch_target = next(iter(train_loader))

print("Input shape:", batch_input.shape)
print("Target shape:", batch_target.shape)

Input shape: torch.Size([32, 32])
Target shape: torch.Size([32, 32])


In [ ]:
import torch
import torch.nn as nn
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)
        return output

In [ ]:
x = torch.randn(32, 32, 128)  # (batch, seq_len, d_model)

attn = ScaledDotProductAttention(128)
out = attn(x)

print(out.shape)

torch.Size([32, 32, 128])


In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Linear layers
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        # Split into heads
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention calculation
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)

        out = torch.matmul(weights, V)

        # Combine heads
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        out = self.fc_out(out)

        return out

In [ ]:
x = torch.randn(32, 32, 128)

mha = MultiHeadAttention(128, 4)
out = mha(x)

print(out.shape)

torch.Size([32, 32, 128])


In [ ]:
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [ ]:
x = torch.randn(32, 32, 128)

ffn = FeedForward(128, 512)
out = ffn(x)

print(out.shape)

torch.Size([32, 32, 128])


In [ ]:
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Attention block
        attn_out = self.attn(x)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed Forward block
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [ ]:
x = torch.randn(32, 32, 128)

block = TransformerBlock(128, 4, 512)
out = block(x)

print(out.shape)

torch.Size([32, 32, 128])


In [ ]:
CONFIG = {
    'vocab_size': vocab_size,  # your vocab (≥4000)
    'd_model': 128,
    'num_heads': 4,
    'num_layers': 2,
    'd_ff': 512,
    'max_seq_length': 32,
    'dropout': 0.1
}

In [ ]:
import torch
import torch.nn as nn
import math

class Transformer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.d_model = config['d_model']
        self.vocab_size = config['vocab_size']
        self.max_seq_length = config['max_seq_length']

        # Embedding
        self.token_embedding = nn.Embedding(self.vocab_size, self.d_model)

        # Positional Encoding
        self.pos_encoding = self.create_positional_encoding()

        # Transformer Blocks
        self.layers = nn.ModuleList([
            TransformerBlock(
                config['d_model'],
                config['num_heads'],
                config['d_ff'],
                config['dropout']
            )
            for _ in range(config['num_layers'])
        ])

        self.dropout = nn.Dropout(config['dropout'])

        # Output layer
        self.fc_out = nn.Linear(self.d_model, self.vocab_size)

    def create_positional_encoding(self):
        pe = torch.zeros(self.max_seq_length, self.d_model)

        for pos in range(self.max_seq_length):
            for i in range(0, self.d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i / self.d_model)))
                if i + 1 < self.d_model:
                    pe[pos, i + 1] = math.cos(pos / (10000 ** (i / self.d_model)))

        return pe.unsqueeze(0)  # shape (1, seq_len, d_model)

    def forward(self, x):
        batch_size, seq_len = x.shape

        # Embedding
        x = self.token_embedding(x)

        # Add positional encoding
        x = x + self.pos_encoding[:, :seq_len, :].to(x.device)

        x = self.dropout(x)

        # Pass through transformer blocks
        for layer in self.layers:
            x = layer(x)

        # Output projection
        out = self.fc_out(x)

        return out

In [ ]:
model = Transformer(CONFIG)

batch = torch.randint(0, vocab_size, (32, 32))
output = model(batch)

print(output.shape)

torch.Size([32, 32, 5001])


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print(count_parameters(model))

1681801


In [ ]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train(model, train_loader, val_loader, epochs=3):
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_input, batch_target in train_loader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)

            optimizer.zero_grad()

            output = model(batch_input)

            # reshape for loss
            output = output.view(-1, vocab_size)
            batch_target = batch_target.view(-1)

            loss = criterion(output, batch_target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

In [ ]:
train(model, train_loader, None, epochs=3)

Epoch 1, Loss: 0.8852
Epoch 2, Loss: 0.1858
Epoch 3, Loss: 0.1658


In [ ]:
def evaluate(model, val_loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch_input, batch_target in val_loader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)

            output = model(batch_input)

            loss = criterion(
                output.view(-1, vocab_size),
                batch_target.view(-1)
            )

            total_loss += loss.item()

    return total_loss / len(val_loader)

In [ ]:
best_val_loss = float('inf')

def train(model, train_loader, val_loader, epochs=5):
    global best_val_loss   # to update outside variable

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch_input, batch_target in train_loader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)

            optimizer.zero_grad()
            output = model(batch_input)

            loss = criterion(
                output.view(-1, vocab_size),
                batch_target.view(-1)
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ✅ VALIDATION
        val_loss = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {avg_loss:.4f}")
        print(f"Val Loss: {val_loss:.4f}")

        # ✅ SAVE BEST MODEL (CORRECT PLACE)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            print("✅ Model Saved!")

In [ ]:
train(model, train_loader, val_loader, epochs=5)

Epoch 1
Train Loss: 0.1577
Val Loss: 0.9628
✅ Model Saved!
Epoch 2
Train Loss: 0.1527
Val Loss: 1.0722
Epoch 3
Train Loss: 0.1487
Val Loss: 1.0066
Epoch 4
Train Loss: 0.1458
Val Loss: 1.0579
Epoch 5
Train Loss: 0.1430
Val Loss: 1.0516


In [ ]:
def predict_next_word(model, text, word2idx, idx2word, max_len=32, top_k=5):
    model.eval()

    tokens = tokenize(text)
    encoded = [word2idx.get(w, 0) for w in tokens]

    encoded = encoded[-max_len:]
    encoded = [0]*(max_len - len(encoded)) + encoded

    x = torch.tensor([encoded]).to(device)

    with torch.no_grad():
        output = model(x)

    probs = torch.softmax(output[0, -1], dim=0)

    topk = torch.topk(probs, top_k * 3)

    results = []
    for i in range(len(topk.indices)):
        word = idx2word[topk.indices[i].item()]
        prob = topk.values[i].item()

        if word.isalpha():
            results.append((word, prob))

        if len(results) == top_k:
            break

    return results

In [ ]:
print(predict_next_word(model, "to be or not to", word2idx, idx2word))

[('me', 0.04387473315000534), ('the', 0.03742552548646927), ('make', 0.0357755683362484), ('be', 0.021268200129270554), ('thee', 0.01862897351384163)]


In [ ]:
train(model, train_loader, val_loader, epochs=10)

Epoch 1
Train Loss: 0.1409
Val Loss: 1.0551
Epoch 2
Train Loss: 0.1388
Val Loss: 1.0678
Epoch 3
Train Loss: 0.1370
Val Loss: 1.0169
Epoch 4
Train Loss: 0.1351
Val Loss: 1.0309
Epoch 5
Train Loss: 0.1335
Val Loss: 1.0764
Epoch 6
Train Loss: 0.1317
Val Loss: 1.0483
Epoch 7
Train Loss: 0.1302
Val Loss: 1.0403
Epoch 8
Train Loss: 0.1286
Val Loss: 1.0435
Epoch 9
Train Loss: 0.1273
Val Loss: 1.0256
Epoch 10
Train Loss: 0.1259
Val Loss: 0.9725


In [ ]:
# STEP 1: Create model
model = Transformer(CONFIG)

# STEP 2: Move to device (CPU/GPU)
model = model.to(device)

# STEP 3: Load best trained weights
model.load_state_dict(torch.load("best_model.pt"))

# STEP 4: Set model to evaluation mode
model.eval()

# STEP 5: Run prediction
print(predict_next_word(model, "to be or not to", word2idx, idx2word))

[('the', 0.08827348053455353), ('me', 0.04291204735636711), ('be', 0.021178554743528366), ('my', 0.018777864053845406), ('him', 0.01852383278310299)]


In [ ]:
import math

val_loss = 0.9725   # from your last output
perplexity = math.exp(val_loss)

print("Perplexity:", perplexity)

Perplexity: 2.6445475709529163


In [ ]:
def compute_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)

            preds = out.argmax(dim=-1)

            correct += (preds == y).sum().item()
            total += y.numel()

    return correct / total

In [ ]:
 acc = compute_accuracy(model, val_loader)
 print("Accuracy:", acc)

Accuracy: 0.9410506905210295


In [ ]:
import os
os.makedirs("project", exist_ok=True)


In [ ]:
open("/content/project/model.py", "w").close()
open("/content/project/train.py", "w").close()
open("/content/project/predict.py", "w").close()
open("/content/project/utils.py", "w").close()
open("/content/project/README.md", "w").close()
open("/content/project/requirements.txt", "w").close()

In [ ]:
import shutil
shutil.copy("/content/best_model.pt", "/content/project/best_model.pt")

'/content/project/best_model.pt'

In [ ]:
!zip -r /content/project.zip /content/project

  adding: content/project/ (stored 0%)
  adding: content/project/utils.py (stored 0%)
  adding: content/project/model.py (stored 0%)
  adding: content/project/.ipynb_checkpoints/ (stored 0%)
  adding: content/project/train.py (stored 0%)
  adding: content/project/predict.py (stored 0%)
  adding: content/project/README.md (stored 0%)
  adding: content/project/requirements.txt (stored 0%)
  adding: content/project/best_model.pt (deflated 7%)
